HW2 full run, frozen prompts and model: 767 assets x 2 prompts (p_para, p_infer), Qwen/Qwen3-4B-Instruct-2507 revision cdbee75f17c01a7cc42f958dc650907174af0554. Metrics on the same demo-100 users with one protocol_hash: p_para, p_infer, original_text_cbf and 4 baselines. After a disconnect, rerun cells 1-3 to restore the cache from Google Drive, then cells 4-8; the run command already uses --resume. Do not rerun after outputs appear.

In [ ]:
!pip -q install "transformers==4.57.6"

In [ ]:
import sys, torch, transformers
print('python', sys.version.split()[0])
print('torch', torch.__version__)
print('transformers', transformers.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('compute_capability', torch.cuda.get_device_capability(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil
DRIVE = Path('/content/drive/MyDrive/hw2_full_run')
DRIVE.mkdir(parents=True, exist_ok=True)
LOCAL = Path('/content/full_run')
(LOCAL / 'generation_cache').mkdir(parents=True, exist_ok=True)
if (DRIVE / 'generation_cache').is_dir():
    shutil.copytree(DRIVE / 'generation_cache', LOCAL / 'generation_cache', dirs_exist_ok=True)
for name in ('generation_log.jsonl', 'failures.jsonl', 'progress.json'):
    source = DRIVE / name
    if source.exists():
        shutil.copy2(source, LOCAL / name)
print('restored_cache_files', len(list((LOCAL / 'generation_cache').glob('*.json'))))

In [ ]:
from google.colab import files
uploaded = files.upload()
expected = ['encoder.py', 'data_adapter.py', 'runtime.py', 'full_run.py', 'starter.py',
            'transactions.csv', 'asset_information.csv',
            'customer_information.csv', 'close_prices.csv']
print('missing', [name for name in expected if name not in uploaded])

In [ ]:
from pathlib import Path
import shutil
base = Path('/content')
(base / 'my_llm').mkdir(exist_ok=True)
(base / 'FAR-Trans').mkdir(exist_ok=True)
(base / 'starter' / 'far-trans').mkdir(parents=True, exist_ok=True)
for name in ['encoder.py', 'data_adapter.py', 'runtime.py', 'full_run.py']:
    shutil.copy2(str(base / name), str(base / 'my_llm' / name))
for name in ['transactions.csv', 'asset_information.csv',
             'customer_information.csv', 'close_prices.csv']:
    shutil.copy2(str(base / name), str(base / 'FAR-Trans' / name))
shutil.copy2(str(base / 'starter.py'), str(base / 'starter' / 'far-trans' / 'starter.py'))

In [ ]:
!python /content/my_llm/full_run.py \
  --data-dir /content/FAR-Trans \
  --starter-dir /content/starter/far-trans \
  --output-dir /content/full_run \
  --model Qwen/Qwen3-4B-Instruct-2507 \
  --revision cdbee75f17c01a7cc42f958dc650907174af0554 \
  --device cuda --allow-download \
  --resume --drive-dir /content/drive/MyDrive/hw2_full_run --save-every 50

In [ ]:
import json
from collections import Counter
from pathlib import Path
out = Path('/content/full_run')
rows = [json.loads(line) for line in (out / 'generation_log.jsonl').read_text(encoding='utf-8').splitlines()]
counts = Counter(r['prompt_mode'] for r in rows)
print('records', len(rows), 'assets', len({r['asset_id'] for r in rows}), 'modes', dict(sorted(counts.items())))
assert len(rows) == 1534
assert len({r['asset_id'] for r in rows}) == 767
assert counts == {'para': 767, 'infer': 767}
metrics = json.loads((out / 'metrics.json').read_text(encoding='utf-8'))
manifest = json.loads((out / 'run_manifest.json').read_text(encoding='utf-8'))
print('protocol_hash', metrics['protocol_hash'])
assert metrics['protocol_hash'] == '50efd4aeb415ba6f0c66dd0793a7c2033cdc32a42b86b58eebbca3e81eaf3417'
print('users', len(manifest['users']), 'group_users', len(manifest['group_users']))
assert len(manifest['users']) == 100 and len(manifest['group_users']) == 54
print('model', manifest['runtime']['model'], 'revision', manifest['runtime']['resolved_revision'])
print('fallbacks', metrics['fallback_counts'])
print((out / 'ndcg_table.csv').read_text(encoding='utf-8'))

In [ ]:
from google.colab import files
from pathlib import Path
out = Path('/content/full_run')
names = ['metrics.json', 'ndcg_table.csv', 'per_user.csv', 'predictions.csv',
         'run_manifest.json', 'generation_log.jsonl', 'failures.jsonl']
for name in names:
    files.download(str(out / name))